# Сборный проект: Carsharing

## Описание проекта

Cоздать систему, которая могла бы оценить риск ДТП по выбранному маршруту движения. 

Под риском понимается вероятность ДТП с любым повреждением транспортного средства. Как только водитель забронировал автомобиль, сел за руль и выбрал маршрут, система должна оценить уровень риска. 

Если уровень риска высок, водитель увидит предупреждение и рекомендации по маршруту.

## Настройка окружения

### Импорты

In [ ]:
%pip install -q phik
%pip install -q optuna
%pip install -q python-Levenshtein
%pip install -Uq scikit-learn
%pip install -Uq shap

In [ ]:
import os

import shap

import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import pandas as pd
import psycopg2
from sqlalchemy import create_engine

from phik import phik_matrix

import plotly.express as px
import plotly.graph_objects as go

import optuna
import OptunaSearchCV

import math

import warnings

from Levenshtein import distance
from collections import defaultdict, Counter

from catboost import CatBoostClassifier

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import make_scorer, f1_score, classification_report, confusion_matrix, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

### Настройки отображения

In [ ]:
# output settings
warnings.filterwarnings("ignore")

options = {
    "display.max_rows": None,
    "display.max_columns": None,
    "display.float_format": "{:,.2f}".format,
    "display.max_colwidth": None,
}

# Применяем опции через цикл
for option, value in options.items():
    pd.set_option(option, value)

### Объявление функций

In [ ]:
def check_size(current_df: pd.DataFrame, original_df: pd.DataFrame):
    print("Количество записей в текущем датасете: {}".format(len(current_df)))
    print("Количество записей в оригинальном датасете: {}".format(len(original_df)))
    print("Процент от начального объема данных: {:.2%}".format(len(current_df) / len(original_df)))

In [ ]:
HOST = "https://code.s3.yandex.net"
HTTP_PREFIX = "http"


def get_dataset_path(dataset_path: str):
    # check server request --> relative path --> absolute path --> yandex server request
    path = (
        dataset_path
        if dataset_path.startswith(HTTP_PREFIX)
        else "." + dataset_path if os.path.exists("." + dataset_path)
        else dataset_path if os.path.exists(dataset_path)
        else HOST + dataset_path
    )
    print("Dataset path:", path)
    return path


# load csv
def load_csv(dataset_path: str, **kwargs):
    path = get_dataset_path(dataset_path)
    try:
        return pd.read_csv(filepath_or_buffer=path, **kwargs)
    except Exception as ex:
        print("Could not load csv. Exception:", str(ex))

In [ ]:
def describe_dataset(df):
    """
    Выводит общую информацию и статистику по датасету pandas.

    Параметры:
    - df (pandas.DataFrame): Исходный датасет.
    """
    # 1. Общая информация о датасете
    print("*** Общая информация ***\n")
    print(df.info())
    print("\n")

    # 2. Статистика по количественным данным
    print("*** Статистика по количественным данным ***")
    display(df.describe())
    print("\n")

   # 3. Статистика по категориальным данным
    print("*** Статистика по категориальным данным ***")
    category_cols = df.select_dtypes(include=['object', 'category']).columns
    if len(category_cols) > 0:
        display(df[category_cols].describe())
    else:
        print("Нет категориальных признаков для анализа.")
    print("\n")

    # 4. Информация по колонкам
    print("*** Информация по колонкам ***")
    column_info = pd.DataFrame(
        {
            "type": df.dtypes,
            "na_count": df.isna().sum(),
            "empty_count": (df == "").sum(),
            "unique_count": df.nunique(),
        }
    )
    display(column_info)
    print("\n")

    # 5. Количество дубликатов
    duplicates = df.duplicated().sum()
    print(f"*** Количество дубликатов: {duplicates} ***")
    print("\n")

    # 6. Первые 5 строк датасета
    print("*** Первые 5 строк датасета ***")
    display(df.head())

In [ ]:
def run_sql(db_config: dict, sql: str) -> pd.DataFrame:
    try:
        conn = psycopg2.connect(**db_config)
        df = pd.read_sql_query(sql, con=conn)
        return df
    except Exception as e:
        print(f"Ошибка при выполнении запроса: {e}")
        raise e
    finally:
        if conn is not None:
            conn.close()

In [ ]:
# alternative way to load some data from db
#
# def load_data_from_db(db_config, table_name):
#     try:
#         uri = f"postgresql://" \
#               f"{db_config['user']}:{db_config['password']}@" \
#               f"{db_config['host']}:{db_config['port']}/" \
#               f"{db_config['dbname']}"
        
#         engine = create_engine(uri)
        
#         df = pd.read_sql_table(table_name, engine)
        
#         print(f"Таблица '{table_name}' успешно загружена.")
#         return df
    
#     except Exception as e:
#         print(f"Ошибка при загрузке данных: {e}")
#         raise
    
#     finally:
#         # закрываем соединение в бд
#         if hasattr(engine, 'pool'):
#             engine.dispose()

## Подключитесь к базе. Загрузите таблицы sql

In [ ]:
db_config = {
    'user': 'praktikum_student',
    'password': 'Sdf4$2;d-d30pp',
    'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
    'port': 6432,
    'dbname': 'data-science-vehicle-db',
}

In [ ]:
df_case_ids = run_sql(db_config, "select * from case_ids limit 10")
df_collisions = run_sql(db_config, "select * from collisions limit 10")
df_parties = run_sql(db_config, "select * from parties limit 10")
df_vehicles = run_sql(db_config, "select * from vehicles limit 10")

### Промежуточные выводы

- Подключение к базе данных прошло успешно
- Таблицы загружены с использованием SQL

## Проведите первичное исследование таблиц

In [ ]:
df_dict = {
    "df_case_ids": df_case_ids,
    "df_collisions": df_collisions,
    "df_parties": df_parties,
    "df_vehicles": df_vehicles,
}

In [ ]:
for df_name, df in df_dict.items():
    print(f"{'-' * 10} start: {df_name} {'-' * 10}\n")
    display(df)
    print(f"\n{'-' * 10} end: {df_name} {'-' * 10}\n")

### Промежуточные выводы

- Все таблицы имеют наборы данных
- Таблица `case_ids` на первый взгляд не представляет интереса для анализа и построения моделей
- Таблицы `collisions`, `parties` и `vehicles` можно использовать для решения поставленной задачи
- Все таблицы содержат общее поле-ключ `case_id`
- Для объединения таблиц `parties` и `vehicles` нужно использовать дополнительное поле `party_number`
- Данные содержат пропуски
- Большинство признаков в данных категориальные

##  Проведите статистический анализ факторов ДТП

Найдем полноту представленных данных за каждый год

In [ ]:
sql = """
SELECT 
    EXTRACT(YEAR FROM collision_date)::int AS year,
    MIN(collision_date) AS min_date,
    MAX(collision_date) AS max_date
FROM collisions
GROUP BY year
ORDER BY year
"""

df_year_dates = run_sql(db_config, sql)
df_year_dates

Дополнительно посмотрим по месяцам каждого года

In [ ]:
sql = """
SELECT 
    EXTRACT(YEAR FROM collision_date)::int AS year,
    EXTRACT(MONTH FROM collision_date)::int AS month,
    COUNT(1) AS total
FROM collisions
GROUP BY year, month
ORDER BY year, month
"""

df_year_month_dates = run_sql(db_config, sql)
df_year_month_dates_pivot_table = df_year_month_dates.pivot(index='year', columns='month', values='total')
df_year_month_dates_pivot_table

In [ ]:
fig = px.line(df_year_month_dates, x='month', y='total', color='year',
              title="Количество ДТП по месяцам и годам",
              labels={'month': 'Месяц', 'total': 'Общее количество ДТП'})

fig.show()

Видно, что наиболее полно представлены данные до мая 2012. После 05.2012 происходит резкое снижение количества записей за каждый месяц.

Для дальнейшего анализа будем рассматривать данные только за полные года (2009-2011), т.к. данные за неполные года искажают средние значения и влияют на выводы

Выясним, в какие месяцы происходит наибольшее количество аварий

In [ ]:
sql = """
SELECT EXTRACT(MONTH FROM collision_date)::int AS month_number,
       COUNT(1) AS accident_count
FROM collisions
WHERE EXTRACT(YEAR FROM collision_date)::int BETWEEN 2009 AND 2011
GROUP BY month_number
"""

df_accidents_grouped_by_month = run_sql(db_config, sql)
df_accidents_grouped_by_month

In [ ]:
fig = px.bar(df_accidents_grouped_by_month, x='month_number', y='accident_count',
             title='Количество аварий по месяцам (2009—2011)',
             labels={'month_number': 'Номер месяца', 'accident_count': 'Количество аварий'},
            )

fig.show()

In [ ]:
fig = px.line(df_accidents_grouped_by_month, x='month_number', y='accident_count',
              title='Динамика аварий по месяцам (2009—2011)',
              labels={'month_number': 'Номер месяца', 'accident_count': 'Количество аварий'})

fig.show()

За период 2009-2011 самое большое количество ДТП происходило в октябре (~112K), а меньше всего ДТП происходило в феврале (~98.4K)

### Задачи для коллег

---
Задача №1: Анализ влияния погодных условий на частоту аварий
Описание: Определите, насколько погодные условия влияют на количество дорожно-транспортных происшествий (ДТП). Необходимо выяснить, какие погодные условия чаще всего приводят к авариям.

Решение включает:

- Объединение таблиц collisions и parties.
- Фильтрация по полю weather_1 в таблице collisions.
- Подсчет количества случаев ДТП по каждой категории погодных условий.

---
Задача №2: Оценка роли нарушения ПДД в причинах ДТП

Описание: Выясните, какие нарушения Правил дорожного движения наиболее часто становятся причиной аварии. Для этого потребуется определить, какое нарушение является доминирующим среди водителей, признанных виновниками столкновений.

Решение включает:

- Соединение таблиц parties и collisions.
- Использование полей at_fault и pcf_violation_category для выявления нарушений, ставших причиной ДТП.

---
Задача №3: Изучение связи возраста автомобиля и серьезности столкновения

Описание: Исследуйте влияние возраста автомобилей на тяжесть последствий ДТП. Есть ли корреляция между возрастом транспортных средств и уровнем повреждения?

Решение включает:

- Связывание таблиц vehicles и collisions.
- Поле vehicle_age из таблицы vehicles сравнивается с полем collision_damage из таблицы collisions.

---
Задача №4: Анализ воздействия алкогольного опьянения водителя на последствия ДТП

Описание: Проанализируйте, как состояние трезвости участников влияет на уровень ущерба и тяжести происшествия. Например, водители, находившиеся в состоянии алкогольного опьянения, могли иметь больше серьезных аварий.

Решение включает:

- Совместное использование таблиц parties и collisions.
- Применение поля party_sobriety и связанного с ним столбца collision_damage.

---
Задача №5: Исследование зависимости между типом автомобиля и характером дорожных столкновений

Описание: Нужно установить, какой тип транспортного средства (легковой автомобиль, грузовик, мотоцикл и др.) чаще участвует в определенных видах аварий (например, лобовые столкновения или столкновение с неподвижным объектом).

Решение включает:

- Свяжите таблицу vehicles с таблицей collisions.
- Используйте поле type_of_collision вместе с полями типа транспорта из таблицы vehicles.

---
Задача №6: Определение наиболее опасных комбинаций объектов и транспортных средств в ДТП

Описание: Выявление наиболее распространенных сочетаний типа транспортного средства и другого участника (или объекта) дорожного движения, участвующих в авариях. Это позволит лучше понять потенциальные опасности на дорогах и меры профилактики.

Решение включает:

- Присоединение всех трёх таблиц (collisions, parties, vehicles).
- Анализ количества сочетаний vehicle_type и party_type 

### Решение двух задач из списка

Решим задачи 1 и 2

Анализ влияния погодных условий на частоту аварий

Порядок решения:

- Создать sql-запрос;
- Построить график;
- Сделайть вывод.

In [ ]:
sql = """
SELECT 
    weather_1 AS weather_condition,
    COUNT(*) AS number_of_collisions
FROM 
    collisions
GROUP BY 
    weather_1
ORDER BY 
    number_of_collisions DESC
"""

df_accidents_grouped_by_weather = run_sql(db_config, sql)
df_accidents_grouped_by_weather

In [ ]:
fig = px.bar(df_accidents_grouped_by_weather, 
             x='weather_condition', 
             y='number_of_collisions', 
             title='Количество аварий по погодным условиям',
             labels={
                 'weather_condition': 'Погода',
                 'number_of_collisions': 'Количество аварий'
             },
            )

fig.show()

<div class="alert alert-danger">
<s><b>😔 Необходимо исправить:</b> Здесь тоже пайчарт не подходит, некоторые категории невозможно рассмотреть</s>
</div>

<div class="alert alert-info"> <b>🎓 Комментарий студента:</b>
Убрал, оставил только bar chart
</div>

<div class="alert alert-success">
<b>👍 Успех:</b> Все верно!
</div>

Большинство ДТП произошло в ясную погоду

Оценка роли нарушения ПДД в причинах ДТП

Порядок решения:

- Создать sql-запрос;
- Построить график;
- Сделайть вывод.

In [ ]:
sql = """
SELECT 
    c.pcf_violation_category AS violation_category,
    COUNT(*) AS count_of_violations
FROM 
    parties p
JOIN 
    collisions c ON p.case_id = c.case_id
WHERE 
    p.at_fault = 1
GROUP BY 
    c.pcf_violation_category
ORDER BY 
    count_of_violations DESC;
"""

df_accidents_grouped_by_violation = run_sql(db_config, sql)
df_accidents_grouped_by_violation

In [ ]:
fig = px.bar(df_accidents_grouped_by_violation, 
             x='violation_category', 
             y='count_of_violations', 
             title='Частота нарушений ПДД по категориям',
             labels={
                 'violation_category': 'Категория нарушения',
                 'count_of_violations': 'Количество нарушений'
             },
            )

# Настраиваем угол поворота подписей, чтобы цифры стали горизонтальными
fig.update_traces(textposition='outside', textfont_size=12, textangle=0)

fig.show()

Больше всего ДТП произошло из-за нарушения скоростного режима

## Создайте модель для оценки водительского риска

### Набор данных на основе первичного предположения заказчика

- Выберите тип виновника — только машина (car).
- Возьмите случаи, когда ДТП привело к любым значимым повреждениям автомобиля любого из участников — все, кроме типа SCRATCH (царапина).
- Для моделирования возьмите данные только за 2012 год.
- Подготовка исходной таблицы должна проводиться с помощью sql-запроса.

In [ ]:
list(df_collisions["collision_damage"].unique())

In [ ]:
list(df_parties["party_type"].unique())

In [ ]:
sql = """
SELECT 
    c.*, 
    p.party_number,
    p.party_type,
    p.AT_FAULT,
    p.insurance_premium,
    p.party_drug_physical,
    p.party_sobriety,
    p.cellphone_in_use,
    v.vehicle_type,
    v.vehicle_transmission,
    v.vehicle_age
FROM 
    collisions c
JOIN 
    parties p ON c.case_id = p.case_id
JOIN 
    vehicles v ON p.case_id = v.case_id AND p.party_number = v.party_number
WHERE 
    p.party_type = 'car' 
AND 
    c.collision_damage <> 'scratch' 
AND 
    EXTRACT(YEAR FROM c.collision_date) = 2012;
"""

df_collisions_full = run_sql(db_config, sql)
df_collisions_full.head()

Общий датасет готов

In [ ]:
describe_dataset(df_collisions_full)

- В датасете содержится 56248 записей
- case_id нельзя сделать индексом, т.к. после объединения таблиц встречаются повторяющиеся значения. Но с точки зрения модели оно и не представляет интереса, его можно удалить
- в данных есть выбросы (max distance = 1584000.00, max vehicle_age = 161) и пропуски
- некорректно определены типы некоторых полей. Например, intersection. Хоть в нем и цифры, но взаимосвязи между значениями нет, это категориальный признак
- некорректно определены типы полей с датами
- некоторые признаки имеют некорректные имена (содержат `_1` в имени), переиминуем их

### Первичный отбор факторов для модели

Нужно отобрать те, которые могут влиять на вероятность ДТП, и аргументировать выбор

In [ ]:
# найдем список всех полей
df_collisions_full.columns.to_list()

Отберем признаки, которые известны на момент начала движения

In [ ]:
columns = [
    'county_location',
    'weather_1',                # Погода. Доступны прогнозы и есть влияние на вероятность ДТП.
    'road_surface',             # Состояние дороги, влияет на вероятность ДТП.
    'road_condition_1',         # Дорожное состояние, может влиять на вероятность ДТП.
    'lighting',                 # Освещение, влияет на вероятность ДТП.
    'control_device',
    'collision_date',           # Дата происшествия. В зависимости от месяца меняются погодные условия, которые могут влиять на ДТП
    'at_fault',                 # Виновность участника. Целевой признак
    'cellphone_in_use',         # Наличие телефона в автомобиле (возможности разговаривать по громкой связи). Может влиять на вероятность ДТП.
    'vehicle_type',
    'vehicle_transmission',     # Тип КПП, может влиять на вероятность ДТП.
    'vehicle_age'               # Возраст автомобиля (в годах). Требование заказчика. Таблица vehicles
 ]

- Нет возможности оценить состояние водителя на момент старта поездки (алкогольное опьянение, прием лекарств)
- Нет возможности оценить стиль вождения и потенциальные нарушения правил со стороны водителя
- Нет смысла брать все признаки, характеризующие случившееся ДТП (например, поле `Дополнительные участники ДТП`)
- Сумма страховой выплаты известна уже после ДТП
- Признаки `collision_time` и `lighting` в какой-то степени дублируют друг друга. Оставим только `lighting`

In [ ]:
df_collisions_data = df_collisions_full[columns]
df_collisions_data.head()

In [ ]:
# rename columns
new_columns = {'weather_1': 'weather', 'road_condition_1': 'road_condition'}
df_collisions_data.rename(columns=new_columns, inplace=True)

# change column types
df_collisions_data['cellphone_in_use'] = df_collisions_data['cellphone_in_use'].astype(object)
df_collisions_data['at_fault'] = df_collisions_data['at_fault'].astype(bool)

# extract month from date
df_collisions_data['collision_date'] = pd.to_datetime(df_collisions_data['collision_date'])
df_collisions_data['collision_month'] = df_collisions_data['collision_date'].dt.month.astype(object)
df_collisions_data = df_collisions_data.drop(columns=["collision_date"])

df_collisions_data.info()

Проведем исследовательский анализ данных полученного датасета

#### Неявные дубликаты

Функции для проверки наличия наявных дубликатов в категориальных признаках.

In [ ]:
def normalize_string(s):
    try:
        if isinstance(s, str):
            cleaned_str = ''.join(c for c in s if c.isalnum() or c.isspace()).strip().lower()
            return cleaned_str
        else:
            return None
    except Exception as e:
        return s


def find_implicit_duplicates(dataframe):
    results = []

    category_cols = dataframe.select_dtypes(exclude=['number']).columns
    
    for column_name in category_cols:
        values = dataframe[column_name].unique().tolist()
        
        # Отфильтруем None-значения и создадим нормализованный список
        filtered_values = [val for val in values if val is not None]
        # Создаем словарь соответствия оригинальных значений и их нормализованных версий
        normalized_values = {v: normalize_string(v) for v in filtered_values}
        
        # Сохраняем оригинальные значения, соответствующие каждой нормализованной форме
        inverse_mapping = defaultdict(list)
        for orig_val, norm_val in normalized_values.items():
            if norm_val is not None:
                inverse_mapping[norm_val].append(orig_val)
        
        # Уникальные нормализованные значения
        unique_normalized_values = list(inverse_mapping.keys())
        
        # Создание групп похожих значений
        similar_groups = defaultdict(list)
        
        # Объединяем похожие нормализованные значения в одну группу
        for norm_val in unique_normalized_values:
            found_group = False
            for existing_group in similar_groups:
                dist = distance(norm_val, existing_group)
                if dist <= len(existing_group) * 0.2:  # Расстояние <= 20%
                    similar_groups[existing_group].extend(inverse_mapping[norm_val])
                    found_group = True
                    break
            if not found_group:
                similar_groups[norm_val] = inverse_mapping[norm_val]
        
        # Формируем итоговый отчет
        result_column = f'{"-" * 10} Столбец: {column_name} {"-" * 10}\n\n'
        has_duplicates = False
        for group_key, group_members in sorted(similar_groups.items(), key=lambda x: len(x[1]), reverse=True):
            # Выбор самого популярного оригинального значения
            most_common_value = Counter(group_members).most_common(1)[0][0]
            similar_values = [v for v in group_members if v != most_common_value]
            
            if similar_values:
                has_duplicates = True
                result_column += f'- Наиболее частое значение: {most_common_value}\nПодобные значения: {similar_values}\n\n'
        
        if not has_duplicates:
            result_column += '- Неявных дубликатов не обнаружено.\n'
        
        results.append(result_column)
    
    return '\n'.join(results)

In [ ]:
print(find_implicit_duplicates(df_collisions_data))

Неявных дубликатов не обнаружено.

Посмотрим на значения категоривальных признаков. Возможно, среди них есть необычные.

In [ ]:
category_cols = df_collisions_data.select_dtypes(exclude=['number']).columns
for column_name in category_cols:
    values = df_collisions_data[column_name].unique().tolist()
    print(f'{"-" * 10} Столбец: {column_name} {"-" * 10}\n{values}\n\n')

В столбце `control_device` встречаются значения `none` и пустое. Очевидно, что это одно и то же значение. Приедем к единому виду

In [ ]:
df_collisions_data['control_device'].replace('none', None, inplace=True)
df_collisions_data['control_device'].unique().tolist()

Пропуски в значениях заполним в пайплайне при подготовке данных к обучению моделей.

#### Промежуточные выводы

- Отобрали признаки, аргументировали выбор
- Переименовали столбцы
- Привели в соответствие типы
- Проверили категориальные признаки на наличие неявных дубликатов

###  Исследовательский анализ признаков

In [ ]:
describe_dataset(df_collisions_data)

Данные содержат пропуски. Заполним их позже в пайплайне, перед обучением модели

#### Количественные признаки

Создадим функцию для визуализации количественных признаков. Фукнция будет строить график распределения и размаха признака.

In [ ]:
def visualize_numerical_distribution_and_range(df, bins_rule="sturges", numeric_cols=None):
    """
    Рисует графики распределения (гистограмма) и размаха (коробчатая диаграмма) для всех числовых признаков.

    Параметры:
    - df (pandas.DataFrame): Исходный датасет.
    - bins_rule (str): Правило для автоматического подбора количества корзин. Возможные значения:
                     'sturges' (правило Стерджеса) или 'sqrt' (корень квадратный).
    """
    # Отбираем только числовые столбцы
    if numeric_cols is None or not list(numeric_cols):
        numeric_cols = df.select_dtypes(include=["number"]).columns

    # Функция для автоматического подбора количества корзин
    def auto_binning(data_length, rule=bins_rule):
        if rule == "sturges":
            return int(math.ceil(math.log2(data_length))) + 1
        elif rule == "sqrt":
            return int(math.sqrt(data_length))
        else:
            raise ValueError("Invalid binning rule provided.")

    # Проходим по каждому числовому столбцу и строим графики
    for col in numeric_cols:
        # Подбор оптимального количества корзин
        optimal_bins = auto_binning(len(df))

        # Гистограмма с боксплотом (график распределения и размаха)
        fig = px.histogram(
            df,
            x=col,
            marginal="box",
            barmode="group",
            nbins=optimal_bins,
            title=f'Распределение и размах признака "{col}"',
        )

        # Визуализация графика
        fig.update_layout(bargap=0.02)
        fig.show()

In [ ]:
numeric_columns = df_collisions_data.select_dtypes(include=['number']).columns
numeric_columns

Построим графики распределения и размаха для количественных признаков.

In [ ]:
visualize_numerical_distribution_and_range(df_collisions_data, "sqrt", numeric_columns)

Наблюдаются явные выбросу по возарсту автомобиля

In [ ]:
df_collisions_data.query("vehicle_age > 20")

Скорее всего, это ошибка в данных.

Даже если такого возраста машины существуют, то это единичные случаи, и такие данные будут только сбивать модель. Удалим их

In [ ]:
df_collisions_data = df_collisions_data.query("(vehicle_age <= 20) | vehicle_age.isna()")
visualize_numerical_distribution_and_range(df_collisions_data, "sqrt", ["vehicle_age"])

#### Категориальные признаки

Создадим функцию для визуализации категориальных признаков. Фукнция будет строить столбчатую диаграмму для признака.

In [ ]:
def visualize_categorical_columns(df, category_cols=None):
    """
    Строит столбчатые и круговые диаграммы для категориальных признаков датасета.
    
    Параметры:
    - df (pandas.DataFrame): Исходный датасет.
    """
    # Получаем список категориальных столбцов
    if category_cols is None or not list(category_cols):
        category_cols = df.select_dtypes(exclude=['number']).columns
    
    # Для каждого категориального столбца строим диаграммы
    for col in category_cols:
        # Частотная таблица для столбца
        freq_table = df[col].value_counts()
        
        # Бар-чарт (столбчатая диаграмма)
        fig_bar = px.bar(freq_table, x=freq_table.index, y=freq_table.values, 
                         labels={"x": f"{col}", "y": "Частота"},
                         title=f"Бар-диаграмма для {col}",
        )
        fig_bar.show()
        
        # # Круговая диаграмма (pie chart)
        # fig_pie = px.pie(names=freq_table.index, values=freq_table.values, 
        #                  title=f"Круговая диаграмма для {col}")
        # fig_pie.show()

In [ ]:
category_columns = df_collisions_data.select_dtypes(exclude=['number']).columns
category_columns

Построим графики для категориальных признаков

In [ ]:
visualize_categorical_columns(df_collisions_data, category_columns)

Проверим полноту датасета, чтобы убедиться в ожидаемом количестве записей.

In [ ]:
check_size(df_collisions_data, df_collisions_full)

#### Промежуточные выводы

- Обнаружены явные выбросы в возрасте автомобиля, удалены из датасета
- Подавляющее большинство записей в выборке пришлось на первую половину года
- Данные очень несбалансированы по категориальным признакам
- Проверили полноту датасета после проведения всех манипуляций

### Корреляционный анализ

Напишем функции для проведения корреляционного анализа данных.

In [ ]:
def correlation_heatmap(df, interval_cols=None):
    """
    Строит матрицу корреляции с использованием phik_matrix и выводит тепловую карту с аннотациями.
    
    Параметры:
    - df (pandas.DataFrame): Исходный датасет.
    """
    # Отбираем только числовые столбцы
    if interval_cols is None or not list(interval_cols):
        interval_cols = df.select_dtypes(include=["number"]).columns

    # Вычисляем матрицу корреляции Phik
    corr_matrix = phik_matrix(df, interval_cols=interval_cols, verbose=False)

    # Определяем оптимальный размер фигуры
    num_cols = len(corr_matrix.columns)
    size_per_col = 0.7  # размер графика на колонку
    figure_width = num_cols * size_per_col
    figure_height = num_cols * size_per_col

    # Вывод тепловой карты
    plt.figure(figsize=(figure_width, figure_height))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm')
    plt.title("Матрица корреляции Phik")
    plt.show()

Построим матрицу корреляции признаков.

In [ ]:
numeric_columns = df_collisions_data.select_dtypes(include=["number"]).columns

correlation_heatmap(df_collisions_data, numeric_columns)

Определим степень зависимости целевого признака от остальных признаков

In [ ]:
def correlation_analysis(corr_matrix, target_column):
    """
    Анализирует матрицу корреляции и выводит степень зависимости целевого признака от остальных признаков.
    
    Параметры:
    - corr_matrix (pandas.DataFrame): Матрица корреляции.
    - target_column (str): Название целевого признака.
    """
    # Получаем строку с корреляциями целевого признака
    correlations = round(corr_matrix[target_column].abs(), 2)
    
    # Исключаем сам целевой признак
    correlations = correlations.drop(target_column)
    
    # Классифицируем зависимости согласно таблице
    very_low_dependency = correlations[(correlations >= 0) & (correlations < 0.2)].index.tolist()
    low_dependency = correlations[(correlations >= 0.2) & (correlations < 0.5)].index.tolist()
    medium_dependency = correlations[(correlations >= 0.5) & (correlations < 0.7)].index.tolist()
    high_dependency = correlations[(correlations >= 0.7) & (correlations < 0.9)].index.tolist()
    very_high_dependency = correlations[(correlations >= 0.9) & (correlations <= 1)].index.tolist()
    
    # Вывод результатов
    print(f"Признак {target_column} имеет:")
    if very_high_dependency:
        print(f"- очень высокую зависимость (> 0.9) от признаков {very_high_dependency}")
    if high_dependency:
        print(f"- высокую зависимость (0.7-0.9) от признаков {high_dependency}")
    if medium_dependency:
        print(f"- среднюю зависимость (0.5-0.7) от признаков {medium_dependency}")
    if low_dependency:
        print(f"- слабую зависимость (0.2-0.5) от признаков {low_dependency}")
    if very_low_dependency:
        print(f"- очень слабую зависимость (0-0.2) от признаков {very_low_dependency}")
    if not (very_high_dependency or high_dependency or medium_dependency or low_dependency or very_low_dependency):
        print("- признаков с достаточной степенью зависимости не найдено.")

In [ ]:
corr_matrix = phik_matrix(df_collisions_data, verbose=False)

correlation_analysis(corr_matrix, "at_fault")

#### Промежуточные выводы

Матрица корреляции показала, что целевой признак `at_fault` имеет очень слабую зависимость от всех признаков и отсутствии линейной взаимосвязи.

Тем не менее, стоит отметить, что тип коробки передач и возраст автомобиля оказывают влияние на аварийность больше других факторов.

### Подготовка данных к обучению моделей

#### Подготовка обучающей и тестовой выборок

Подготовим выборки: обучающую и тестовую.

In [ ]:
RANDOM_STATE = 1
TEST_SIZE = 0.25

X = df_collisions_data.drop(columns=["at_fault"])
y = df_collisions_data["at_fault"]

# Создаем новый комбинированный признак
# combined_feature = X[["weather", "road_surface"]].apply(lambda row: '_'.join(map(str, row)), axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    # stratify=combined_feature,
    test_size=TEST_SIZE,
    shuffle=True,
    random_state=RANDOM_STATE)

print(X.shape, y.shape)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

Будем использовать `StandardScaler` для масштабирования количественных признаков. Категориальные признаки обработаем с помощью `OneHotEncoder`. Пропуски в значениях заполним медианой для количественных признаков и самым часто встреачющимся значением для категориальных признаков.

In [ ]:
# Выделим количественные и категоривальные признаки
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

print(f"Количественные признаки: {num_cols}")
print(f"Категориальные признаки: {cat_cols}")

Проведем предобработку признаков с использованием `ColumnTransformer`.

In [ ]:
# Создадим transformer, который применяет разные препроцессоры к разным признакам
preprocessor = ColumnTransformer(
    transformers=[
        # Числовые признаки:
        ('num', Pipeline([
            ('imputer_num', SimpleImputer(strategy='median')),  # Заполняем пропуски медианой
            ('scaler', StandardScaler())                        # Масштабируем
        ]), num_cols),
        
        # Категориальные признаки:
        ('cat', Pipeline([
            ('imputer_cat', SimpleImputer(strategy='most_frequent')),                           # Заполняем пропуски самым частотным значением
            ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))  # Проводим ohe-кодирование
        ]), cat_cols)
    ],
    remainder='passthrough'  # Остальные признаки оставляем без изменений
)

# Применим preprocessor к тренировочному набору
X_train_preprocessed = preprocessor.fit_transform(X_train)

# Применим preprocessor к тестовому набору
X_test_preprocessed = preprocessor.transform(X_test)

In [ ]:
print("Количество признаков:", X_train_preprocessed.shape[1])

Данные подготовлены к обучению моделей

#### Промежуточные выводы

- Данные были разбиты на обучающую и тестовую выборки
- Проведено масштабирование количественных признаков с использованием `StandardScaler`, категориальные признаки обработаны с помощью `OneHotEncoder`

### Обучение моделей

Учитывая бизнес-задачи и необходимость минимизации как ложноположительных, так и ложноотрицательных случаев, наиболее подходящей метрикой будет F1 Score.

- Баланс точности и полноты: F1 Score учитывает обе важные характеристики модели — способность точно предсказывать наличие ДТП (precision) и способность находить все реальные случаи ДТП (recall).
- Несбалансированные классы: Поскольку большинство маршрутов, вероятно, не связаны с ДТП, использование метрики, чувствительной к балансу классов, критически важно.
- Практический смысл: Высокая оценка F1 Score обеспечит, что водители получат своевременные и точные предупреждения, минимизируя как избыточные уведомления, так и пропуск опасных ситуаций.

Таким образом, F1 Score является оптимальной метрикой для оценки эффективности модели в данной задаче.

Обучим для сравнения 4 модели:

- Logistic Regression
- Decision Tree
- Random Forest
- Neural Network

Дополнительно с помощью Optuna подберем основные гиперпараметры моделей для поиска лучших значений. Также будем будем сравнивать значение метрики на кросс-валидации.

In [ ]:
f1_scorer = make_scorer(f1_score)

#### Logistic Regression

In [ ]:
# Logistic Regression
def objective_lr(trial):
    params = {
        'C': trial.suggest_loguniform('C', 1e-5, 1e+5),
        'solver': trial.suggest_categorical('solver', ['newton-cg', 'lbfgs', 'liblinear']),
        'max_iter': trial.suggest_int('max_iter', 100, 1000),
        'random_state': RANDOM_STATE,
    }
    
    model = LogisticRegression(**params)
    scores = cross_val_score(model, X_train_preprocessed, y_train, cv=5, scoring=f1_scorer)
    return scores.mean()

In [ ]:
study_lr = optuna.create_study(direction='maximize', study_name="Logistic Regression")
study_lr.optimize(objective_lr, n_trials=50)
best_score_lr = study_lr.best_value
best_params_lr = study_lr.best_params
print()
print('F1 Score:', round(best_score_lr, 2))
print('Best hyperparameters:', best_params_lr)

#### Decision Tree

In [ ]:
# Decision Tree
def objective_dt(trial):
    params = {
        'max_depth': trial.suggest_int("max_depth", 2, 32),
        'min_samples_split': trial.suggest_int("min_samples_split", 2, 20),
        'criterion': trial.suggest_categorical("criterion", ["gini", "entropy"]),
        'random_state': RANDOM_STATE,
    }
    
    model = DecisionTreeClassifier(**params)
    scores = cross_val_score(model, X_train_preprocessed, y_train, cv=5, scoring=f1_scorer)
    return scores.mean()

In [ ]:
study_dt = optuna.create_study(direction='maximize', study_name="Decision Tree")
study_dt.optimize(objective_dt, n_trials=50)
best_score_dt = study_dt.best_value
best_params_dt = study_dt.best_params
print()
print('F1 Score:', round(best_score_dt, 2))
print('Best hyperparameters:', best_params_dt)

#### Random Forest

In [ ]:
# Random Forest
def objective_rf(trial):
    params = {
        'n_estimators':trial.suggest_int("n_estimators", 10, 500),
        'max_depth':trial.suggest_int("max_depth", 2, 32),
        'min_samples_split':trial.suggest_int("min_samples_split", 2, 20),
        'random_state': RANDOM_STATE,
    }
    
    model = RandomForestClassifier(**params)
    scores = cross_val_score(model, X_train_preprocessed, y_train, cv=5, scoring=f1_scorer)
    return scores.mean()

In [ ]:
study_rf = optuna.create_study(direction='maximize', study_name="Random Forest")
study_rf.optimize(objective_rf, n_trials=50)
best_score_rf = study_rf.best_value
best_params_rf = study_rf.best_params
print()
print('F1 Score:', round(best_score_rf, 2))
print('Best hyperparameters:', best_params_rf)

#### Neural Network

In [ ]:
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.use_deterministic_algorithms(True)

In [ ]:
# n_in_neurons = 102
n_in_neurons = X_train_preprocessed.shape[1]
n_hidden_neurons_1 = 70
n_hidden_neurons_2 = 30
n_out_neurons = 1

In [ ]:
X_train_tensor = torch.FloatTensor(X_train_preprocessed)
X_test_tensor = torch.FloatTensor(X_test_preprocessed)
y_train_tensor = torch.FloatTensor(y_train.values)
y_test_tensor = torch.FloatTensor(y_test.values)

Создадим функцию для перебора параметров

In [ ]:
class NeuralNet(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, output_dim):
        super(NeuralNet, self).__init__()
        self.fc1 = torch.nn.Linear(input_dim, hidden_dim1)
        self.fc2 = torch.nn.Linear(hidden_dim1, hidden_dim2)
        self.fc3 = torch.nn.Linear(hidden_dim2, output_dim)
        
    def forward(self, x):
        out = F.relu(self.fc1(x))
        out = F.relu(self.fc2(out))
        out = F.sigmoid(self.fc3(out))
        return out

# Определение функций потерь
criterion = torch.nn.BCELoss()

def create_dataloader(X_data, y_data, batch_size):
    dataset = TensorDataset(X_data, y_data)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataloader

def train_and_evaluate(model, optimizer, criterion, X_train, y_train, epochs, batch_size):
    for epoch in range(epochs):
        running_loss = 0.0
        dataloader = create_dataloader(X_train, y_train, batch_size=batch_size)
        for inputs, labels in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.view(-1), labels.to(dtype=torch.float32))
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        # if epoch % 10 == 0 or epoch == epochs - 1:
        #     print(f'Epoch {epoch}, Training Loss: {running_loss / len(dataloader)}')

    with torch.no_grad():
        model.eval()
        val_outputs = model(X_train)
        predicted_labels = (val_outputs > 0.5).to(int)
        return f1_score(y_train, predicted_labels.numpy())

def objective_nn(trial):
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    num_epochs = trial.suggest_int("num_epochs", 10, 100)
    batch_size = trial.suggest_int("batch_size", 1000, 42184)
    
    model = NeuralNet(
        input_dim=n_in_neurons,
        hidden_dim1=n_hidden_neurons_1,
        hidden_dim2=n_hidden_neurons_2,
        output_dim=n_out_neurons
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    f1 = train_and_evaluate(model, optimizer, criterion, X_train_tensor, y_train_tensor, num_epochs, batch_size)
    return f1


Запустим перебор параметров с помощью Optuna

In [ ]:
study_nn = optuna.create_study(direction="maximize", study_name="Neuron Network")
study_nn.optimize(objective_nn, n_trials=50)
best_score_nn = study_nn.best_value
best_params_nn = study_nn.best_params
print()
print('F1 Score:', round(best_score_nn, 2))
print('Best hyperparameters:', best_params_nn)

#### Промежуточные выводы

Показатели метрики `f1 score` моделей на тренировочных данных

| Модель               | F1 Score |
|----------------------|----------|
| Logistic Regression  | 0.66     |
| Decision Tree        | 0.62     |
| Random Forest        | 0.61     |
| Neural Network       | 0.67     |

Лучшей выбрана модель Neural Network. На тестовых данных будем проверять ее. Одинаковые показатели с Logistic Regression связаны с округлением показателей.

### Проверка лучшей модели

In [ ]:
def create_dataloader(X_data, y_data, batch_size, shuffle=True):
    dataset = TensorDataset(X_data, y_data)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return dataloader

In [ ]:
def train_and_evaluate(model, optimizer, criterion, X_train, y_train, epochs, batch_size):
    for epoch in range(epochs):
        running_loss = 0.0
        dataloader = create_dataloader(X_train, y_train, batch_size=batch_size)
        for inputs, labels in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.view(-1), labels.to(dtype=torch.float32))
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f'Epoch {epoch}, Training Loss: {running_loss / len(dataloader)}')

    return model

def get_trained_model(X_train, y_train, learning_rate, num_epochs, batch_size):
    model = NeuralNet(
        input_dim=n_in_neurons,
        hidden_dim1=n_hidden_neurons_1,
        hidden_dim2=n_hidden_neurons_2,
        output_dim=n_out_neurons
    )

    criterion = torch.nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    return train_and_evaluate(model, optimizer, criterion, X_train, y_train, num_epochs, batch_size)

params = study_nn.best_params
# params = {'learning_rate': 2.0483210376053597e-05, 'num_epochs': 13, 'batch_size': 31751}

model_nn = get_trained_model(X_train_tensor, y_train_tensor, **params)
model_nn.eval()
val_outputs = model_nn(X_test_tensor)
predicted_labels = (val_outputs > 0.5).to(int)
f1_score_test = f1_score(y_test_tensor.numpy(), predicted_labels.detach().numpy())
print(f"F1 Score test (NN): {f1_score_test}")


На тестовых данных модель показала схожие результаты

## Анализ важности факторов ДТП

### Shap диаграмма

Найдем имена признаков после предобрабоки в пайплайне

In [ ]:
final_feature_names = preprocessor.get_feature_names_out()

Построим shap диаграмму для лучшей модели

In [ ]:
# Возьмём первые 100 образцов в качестве фоновых данных
dataloader = create_dataloader(X_train_tensor, y_train_tensor, study_nn.best_params["batch_size"])
background = next(iter(dataloader))[0][:100]

In [ ]:
explainer = shap.DeepExplainer(model_nn.cpu(), background.cpu())

In [ ]:
# Test data
testloader = create_dataloader(X_test_tensor, y_test_tensor, study_nn.best_params["batch_size"], shuffle=False)
test_inputs = next(iter(testloader))[0][:100]

# Move to CPU before passing into SHAP
test_inputs_cpu = test_inputs.cpu()

# Calculate SHAP values
shap_values = explainer.shap_values(test_inputs_cpu, check_additivity=False)

In [ ]:
shap.summary_plot(shap_values, test_inputs_cpu.reshape(len(test_inputs_cpu), -1), feature_names=final_feature_names)

На диаграмме все признаки "слились" в один. Не очень информативно.

<div class="alert alert-info"> <b>🎓 Комментарий студента:</b>
И у меня не получилось найти решение этой проблемы. Поэтому анализ важности признаков сделал для модели логистической регресии, которая заняла второе место, согласно метрики оценки качества модели
</div>

Проведем аналогичный анализ важности признаков для модели логистической регрессии. Она показала второй результат по качеству предсказаний. И результат очень близкий к модели нейронной сети. Можно предположить, что порядок важности признаков у них будет одинаковый.

In [ ]:
model_lr = LogisticRegression(random_state=RANDOM_STATE, **study_lr.best_params)
model_lr.fit(X_train_preprocessed, y_train)

# Предсказания классов
predictions = model_lr.predict(X_test_preprocessed)

# Вероятности принадлежности к классам
probabilities = model_lr.predict_proba(X_test_preprocessed)

f1_score_test_lr = f1_score(y_test, predictions)
print(f'F1 Score test (LR): {f1_score_test_lr:.2f}')

In [ ]:
explainer = shap.LinearExplainer(model_lr, X_train_preprocessed)
shap_values = explainer.shap_values(X_test_preprocessed)

In [ ]:
shap.summary_plot(shap_values, X_test_preprocessed, feature_names=final_feature_names)

Основные факторы, влияющие на дтп:

- состояние и технические характеристики автомобиля
- время суток
- погодные условия

### Графический анализ «Матрицы ошибок»

Построим матрицу ошибок

In [ ]:
# Отчёт по классификационным метрикам
report = classification_report(y_test, predictions)
print(report)

# Матрица ошибок
matrix = confusion_matrix(y_test, predictions)

In [ ]:
# Расчет полноты и точности
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)

# Визуализируем матрицу ошибок
plt.figure(figsize=(8, 6))
sns.heatmap(matrix, annot=True, fmt='d', cmap="Blues")
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()

### Диаграмма зависимости

Создадим функцию для построения диаграммы рассеяния (scatter plot) переданного количественного признака в зависимости от значения переданного категориального признака.

In [ ]:
def scatter_plot_by_category(df, quant_col, cat_col):
    """
    Отображает диаграмму рассеяния (scatter plot) количественного признака относительно категориального.
    
    Параметры:
    - df: исходный Dataset (pandas DataFrame).
    - quant_col: название количественного признака (числового типа).
    - cat_col: название категориального признака (строкового типа или факторизированного).
    """
    # Проверяем наличие указанных столбцов в DataFrame
    if quant_col not in df.columns or cat_col not in df.columns:
        raise ValueError(f"В указанном датасете отсутствуют указанные столбцы '{quant_col}', '{cat_col}'.")
        
    # Строим диаграмму рассеяния с группировкой по категориям
    fig = px.scatter(
        data_frame=df,
        x=quant_col, # Количественный признак по оси X
        y=df.index, # Используем индекс строки как координату Y
        color=cat_col, # Цвет точек определяется категорией
        hover_data=[df.index], # Подсказка при наведении мыши включает номер строки
        title=f"{quant_col} vs {cat_col}",
        # width=800, height=600 # Размер графика
    )
    
    # Обновляем разметку осей и заголовков
    fig.update_xaxes(title_text=quant_col)
    fig.update_yaxes(title_text="Наблюдения")
    fig.update_traces(marker=dict(size=8)) # Размер маркера точки
    
    return fig.show()

Построим различные диаграммы зависимости возраста автомобиля и ДТП

In [ ]:
scatter_plot_by_category(df_collisions_data, "vehicle_age", "at_fault")

In [ ]:
fig = px.box(df_collisions_data, x="vehicle_age", y="at_fault", color="at_fault")
fig.update_layout(
    title_text="Распределение ДТП по возрасту автомобиля",
    xaxis_title="Возраст автомобиля",
    yaxis_title="Вероятность ДТП",
    legend_title="Происходило ДТП?"
)
fig.show()

In [ ]:
fig = px.violin(df_collisions_data, x="at_fault", y="vehicle_age", color="at_fault")
fig.update_traces(meanline_visible=True)
fig.update_layout(
    title="Распределение возраста автомобиля по ДТП",
    xaxis_title="Aвария произошла?",
    yaxis_title="Возраст автомобиля"
)
fig.show()

In [ ]:
fig = px.histogram(df_collisions_data, x="vehicle_age", color="at_fault", marginal="rug", histnorm='probability density')
fig.update_layout(
    title="Гистограмма: Среднее значение возраста автомобилей по факту аварии",
    xaxis_title="Средний возраст автомобиля",
    yaxis_title="Плотность вероятности"
)
fig.show()

In [ ]:
# Группировка данных по интервалам возраста автомобиля
bins = pd.cut(df_collisions_data['vehicle_age'], bins=10)
grouped_df = df_collisions_data.groupby(bins)['at_fault'].mean().reset_index()

# Преобразуем объекты Interval в строковое представление
grouped_df['bin'] = grouped_df['vehicle_age'].apply(lambda interval: f"{interval.left:.1f}-{interval.right:.1f}")

# Удаляем исходный столбец vehicle_age
del grouped_df['vehicle_age']

# Рендерим график
fig = px.line(grouped_df, x="bin", y="at_fault", markers=True)
fig.update_layout(
    title="Зависимость частоты аварий от возраста автомобиля",
    xaxis_title="Возраст автомобиля (интервал)",
    yaxis_title="Частота аварий (среднее значение)"
)
fig.show()

In [ ]:
bar_data = df_collisions_data.groupby("vehicle_age").agg({"at_fault": ['mean', 'count']}).reset_index()
bar_data.columns = ['vehicle_age', 'mean_target', 'count']

fig = px.bar(bar_data, x="vehicle_age", y="mean_target")
fig.update_layout(
    title="Средняя вероятность аварии по разному возрасту",
    xaxis_title="Средний возраст автомобиля",
    yaxis_title="Средняя вероятность аварии"
)
fig.show()

Аварии случаются чаще с автомобилями около 3х лет и после 15 лет. Это ожидаемый результат, т.к. гарантия производителя на автомобиль заканчивается обычно через 3 года либо при достижении определенного пробега. А после 15 лет износ деталей автомобиля и риск поломки высокий.

### Промежуточные выводы

Точность (precision) невысокая, модель часто ошибается. А вот значение полноты (recall) высокое, модель предсказывает подавляющее большинство случаев ДТП.

Другими словами, модель "перестраховывается" и "видит" ДТП даже в тех случаях, где предпосылок к этому немного. Т.е. предупреждения о возможности ДТП будут приходить водителям сильно чаще, чем хотелось бы.

## Выводы

В данной работе был проведен анализ возможности предсказания ДТП на основании имеющихся данных о различных факторах.

Было выполнено:

- загрузка и первичное исследование таблиц и данных
- статистический анализ факторов ДТП
- отобраны факторы для анализа риска ДТП, известные на момент начала движения по маршруту
- исследовательский анализ количественных и категориальных признаков
- корреляционный анализ признаков
- подготовка данных к обучению моделей: заполнение пропусков, удаление выбросов, созданы новые признаки
- обучение нескольких моделей для сравнения их результатов, выбраны лучшие
- shap анализ важности факторов ДТП
- графический анализ матрицы ошибок
- графический анализ зависимости целевого признака и гравного фактора

Таблица сравнения моделей

| Модель               | F1 Score |
|----------------------|----------|
| Logistic Regression  | 0.66     |
| Decision Tree        | 0.62     |
| Random Forest        | 0.61     |
| Neural Network       | 0.67     |

Общий вывод:

На вероятность ДТП влияют различные факторы. Основные из них:

- техническое состояние и возраст автомобиля
- погодные и дорожные условия, время суток
- состояние водителя

Погодные и дорожные условия в данных представлены достаточно полно. А вот данных о техническом состоянии автомобиля и состоянии водителя почти нет. Имеющиеся данные о состоянии водителя на момент происшествия использовать для обучения модели нельзя, т.к. нет возможности их оценить на момент начала движения. Также нет информации (истории) об общем поведении водителя на дороге. Например: насколько агрессивно водит, как часто нарушает правила, насколько грубые нарушения и т.д.

Данных о техническом состоянии автомобиля (кроме его возраста и типа коробки передач) не предоставлено.

Как следствие, модель будет чаще необходимого предупреждать водителей о возможности ДТП, что может вызывать раздражение со стороны пользователей сервиса.

Таким образом, чтобы улучшить модель, необходимо дополнительно собирать данные о техническом состоянии автомобиля и состоянии водителя перед началом движения.

Как это сделать:

- Все автомобили должны проходить техническое обслуживание в сервисе. Наверняка, есть возможность собирать данные сервисов обслуживания автомобиля. Помимо этого, (почти) каждый более-менее современный автомобиль содержит бортовой компьютер, который также хранит информацию о техническом состоянии. Можно дополнительно использовать и эти данные тоже.
- Для оценки состояния водителя можно поставить анализаторы алкогольного опьянения и камеру для контроля использования. Помимо этого, можно интегрироваться с современными сервисами здоровья.
- Сохранять историю поведения водителей на дорогах или интегрироваться с базой данных полиции. Это позволит анализировать и учитывать дополнительные факторы поведения водителей. Более того, это будет еще и работать в качестве стоп-фактора для неадекватных водителей. А сами водители будут вести себя аккуратней, если будут знать, что за ними "следят".
- Внедрить современные системы помощи водителю. Например, бортовой комьютер может отслеживать превышение скорости и программно ее снижать.